<a href="https://colab.research.google.com/github/bnsreenu/python_for_microscopists/blob/master/334_training_YOLO_V8_EM_platelets_converted_labels.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

---


In [ ]:
import torch
print(torch.cuda.is_available())

if 'model' in globals():
    del model
    
torch.cuda.empty_cache()
torch.cuda.ipc_collect()

*******Predict******

In [ ]:
import os
from ultralytics import YOLO
%matplotlib inline
from matplotlib import pyplot as plt
from PIL import Image
import torch

#List the saved models in 'runs' directory. Note that you will see multiple 'train' subdirectories numbered 1, 2, 3, etc. The exact number depends on the number of epochs.
#%ls E:\University\Finalproject\BeeCount\Main\Yolo\Results\960\400+patience20_epochs_32+353\weights

#Define a project --> Destination directory for all results
project = "E:\\University\\Finalproject\\BeeCount\\Main\\Yolo\\runs\\segment\\NEW"

#Define subdirectory for this specific training
name = "V.162-เดือน1" 

device = 'cuda' if torch.cuda.is_available() else 'cpu'
my_new_model = YOLO('E:\\University\\Finalproject\\BeeCount\\Main\\Yolo\Results\\960\\full_200_epochs_162_batch_4\\weights\\best.pt').to(device)

new_image = 'X:\\Hive pics\\4th months 11\\9A.JPG'#_DSC5334.JPGd

# Prediction
new_results = my_new_model.predict(new_image, project=project, save=True, name=name, imgsz=960, conf=0.45, show_labels=False, max_det=1000)  #Adjust conf threshold #show_labels=False 



In [ ]:
new_result_array = new_results[0].plot()
plt.figure(figsize=(12, 12)) 
plt.imshow(new_result_array)

-----

Push the mask to cpu (from GPU) and convert to numpy array for easy plotting.

In [ ]:
result = new_results[0]

In [ ]:
new_results

In [ ]:
result.masks.xyn

In [ ]:
extracted_masks = result.masks.data

In [ ]:
extracted_masks.shape

In [ ]:
masks_array = extracted_masks.cpu().numpy()

plt.imshow(masks_array[1])

**Calculating region properties for all objects and saving to a csv file.**

In [ ]:
import pandas as pd
from skimage.measure import regionprops

#**Extracting bounding boxes and segmented masks from the result**
new_result = new_results[0]

new_result

#**Extracting bounding polygons** Use 'Masks.xyn' for segments (normalized) and 'Masks.xy' for segments (pixels)
new_result.masks.xyn

#Extract
extracted_masks = new_result.masks.data

extracted_masks.shape

#**Extracting labels for each class**
class_names = new_result.names.values()
class_names

# Extract the boxes, which likely contain class IDs
detected_boxes = new_result.boxes.data
# Extract class IDs from the detected boxes
class_labels = detected_boxes[:, -1].int().tolist()
# Initialize a dictionary to hold masks by class
masks_by_class = {name: [] for name in new_result.names.values()}

# Iterate through the masks and class labels
for mask, class_id in zip(extracted_masks, class_labels):
    class_name = new_result.names[class_id]  # Map class ID to class name
    masks_by_class[class_name].append(mask.cpu().numpy())

for class_name, masks in masks_by_class.items():
    print(f"Class Name: {class_name}, Number of Masks: {len(masks)}")

#**Extracting masks for a specific class**
cell_honeyuncapped_masks = masks_by_class['open']
cell_honeycapped_masks = masks_by_class['close']
total_masks = masks_by_class['other']

# Extract the original image
orig_img = new_result.orig_img

orig_img.shape

# Initialize a list to store the properties
props_list = []

# Iterate through all classes
for class_name, masks in masks_by_class.items():
    # Iterate through the masks for this class
    for mask in masks:
        # Convert the mask to an integer type if it's not already
        mask = mask.astype(int)

        # Apply regionprops to the mask
        props = regionprops(mask)

        # Extract the properties you want (e.g., area, perimeter) and add them to the list
        for prop in props:
            area = prop.area
            # Add other properties as needed

            # Append the properties and class name to the list
            props_list.append({'Class Name': class_name, 'Area': area})

# Convert the list of dictionaries to a DataFrame
props_df = pd.DataFrame(props_list)

# **Automatically calculate the total area for both "open" and "close" classes**
total_open_area = props_df[props_df['Class Name'] == 'open']['Area'].sum()
total_close_area = props_df[props_df['Class Name'] == 'close']['Area'].sum()
total_other_area = props_df[props_df['Class Name'] == 'other']['Area'].sum()

# **Calculate the combined total area of both "open" and "close"**
total_combined_area = total_open_area + total_close_area
total_all = total_open_area + total_close_area + total_other_area

# ****Save the DataFrame to a CSV file
props_df.to_csv('E:\\University\\Finalproject\\BeeCount\\Main\\Yolo\\Results\\CSV\\V.164-JULY-32A.csv', index=False)

# Print the total areas of "open" and "close" classes, and their combined sum
print(f"Total Area of 'open' class: {total_open_area}")
print(f"Total Area of 'close' class: {total_close_area}")
print(f"Total Combined Area of 'open' and 'close' classes: {total_combined_area}")
print(f"Total All Classes: {total_all}")

props_df


-------

**Swarm plot**

**Box Plot**

**Export model to ONNX for deployment.**